# Phase 2: Data Analysis  

**Data Features:** 
| Column | Values | Meaning | Role |
|---|---|---|---|
| `recency` | 1–12 | Months since last purchase | Feature |
| `history` | $ | Past-year spend | Feature |
| `mens` | 0/1 | Bought men's items in past year | Feature |
| `womens` | 0/1 | Bought women's items in past year | Feature |
| `zip_code` | Urban / Suburban / Rural | Area type. | Feature |
| `newbie` | 0/1 | Became a customer in the past 12 months | Feature |
| `channel` | Phone / Web / Multichannel | Where they bought in the past year | Feature |
| `treatment` | Mens E-Mail / Womens E-Mail / No E-Mail | Randomly assigned group | **Treatment** |
| `visit` | 0/1 | Visited the site within 2 weeks | Outcome |
| `conversion` | 0/1 | Bought within 2 weeks | Outcome |
| `spend` | $ | Amount spent within 2 weeks (0 for non-buyers, not missing) | Outcome |

In [1]:
import duckdb
from digital_sales.paths import DB_PATH

In [2]:
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    df = con.table('analysis_base').df()

# First Look

In [3]:
tdf = df.groupby('treatment')[['visit', 'conversion','spend']].mean().reindex(['no_email','womens_email','mens_email'])
tdf

,visit,conversion,spend
treatment,,,
no_email,0.106167,0.005726,0.652789
womens_email,0.151400,0.008837,1.077202
mens_email,0.182757,0.012531,1.422617


At first look it seems there has been a positive effect on the visit, conversion and spending of customers contacted by either the mens or womens email compared to the control group (no email). We summarize the results in the following bar chart

In [4]:
import plotly.express as px

px.bar(tdf, barmode='group', )

# Validity-Checks  
Before running the statistical analysis on the outcomes we check the experimental groups are fair. To do so we check 2 properties  
1. That the group sizes are similar through a sample ratio mismatch (SRM) 
2. That the groups have similar feature composition, to rule out that some treatment was given preferentially to some type of customers, by using the Covariate Balance.


For the covariate balance we check that the standardised mean difference of each feature is close to 0 (<0.1).
The formula for the SMD is
$$\text{SMD} = \frac{\bar{x}_{\text{email}} - \bar{x}_{\text{no email}}}{\sqrt{\left(s^2_{\text{email}} + s^2_{\text{no email}}\right)/2}}$$

In [5]:
from scipy import stats
import numpy as np
import pandas as pd

# 1. SRM: are the group sizes consistent with the planned 1/3 each?
counts = df["treatment"].value_counts()
chi2, p_srm = stats.chisquare(counts)  # expected: equal thirds
print(f"SRM test: chi2 = {chi2:.2f}, p = {p_srm:.2f}")

# 2. Covariate Balance: standardised mean difference of each pre-campaign feature vs no email
features = pd.get_dummies(
    df[["recency", "history", "mens", "womens", "newbie", "zip_code", "channel"]], dtype=float
)
means = features.groupby(df["treatment"]).mean()
variances = features.groupby(df["treatment"]).var()

# standardized mean difference
smd = pd.DataFrame({
    group: (means.loc[group] - means.loc["no_email"])
           / np.sqrt((variances.loc[group] + variances.loc["no_email"]) / 2)
    for group in ["mens_email", "womens_email"]
})
print(f"Largest |SMD|: {smd.abs().max().max():.3f}  (balanced if < 0.1)")
smd.round(3)


SRM test: chi2 = 0.20, p = 0.90
Largest |SMD|: 0.014  (balanced if < 0.1)


,mens_email,womens_email
recency,0.007,0.005
history,0.008,0.007
mens,-0.005,-0.009
womens,0.008,0.005
newbie,-0.001,0.003
zip_code_Rural,0.014,0.004
zip_code_Suburban,-0.012,-0.001
zip_code_Urban,0.002,-0.002
channel_Multichannel,-0.004,-0.005
channel_Phone,-0.008,0.009


The groups are of equal size (chi2 p-value =0.9) and the features look balanced among treatments (largest SMD=0.014). We can proceed with the analysis

## Lift Analysis
We now check the statistical significance of the differences between groups, and the size of the differences between groups

In [6]:
from statsmodels.stats.multitest import multipletests

PAIRS = [
    ("mens_email", "no_email"),
    ("womens_email", "no_email"),
    ("mens_email", "womens_email"),
]


def values(outcome, group):
    """One outcome for one treatment group, as a numpy array."""
    return df.loc[df["treatment"] == group, outcome].to_numpy(dtype=float)

def proportion_test(y_t, y_r, alpha=0.05):
    """Two-proportion z-test for 0/1 outcomes: treated group t vs reference group r."""
    x_t, n_t = y_t.sum(), y_t.size          # successes, customers
    x_r, n_r = y_r.sum(), y_r.size
    p_t, p_r = x_t / n_t, x_r / n_r
    diff = p_t - p_r
    z_crit = stats.norm.ppf(1 - alpha / 2)  # 1.96

    # Test: SE under H0, where both groups share one pooled rate
    p_pool = (x_t + x_r) / (n_t + n_r)
    se_h0 = np.sqrt(p_pool * (1 - p_pool) * (1 / n_t + 1 / n_r))
    p_value = 2 * stats.norm.sf(abs(diff / se_h0))

    # CI for the difference: each group keeps its own rate
    se = np.sqrt(p_t * (1 - p_t) / n_t + p_r * (1 - p_r) / n_r)

    # CI for the relative lift: on the log of the rate ratio, then back-transformed
    log_rr = np.log(p_t / p_r)
    se_log = np.sqrt((1 - p_t) / x_t + (1 - p_r) / x_r)

    return {
        "treated_value": p_t,
        "reference_value": p_r,
        "abs_lift": diff,
        "abs_ci_low": diff - z_crit * se,
        "abs_ci_high": diff + z_crit * se,
        "rel_lift": p_t / p_r - 1,
        "rel_ci_low": np.exp(log_rr - z_crit * se_log) - 1,
        "rel_ci_high": np.exp(log_rr + z_crit * se_log) - 1,
        "p_value": p_value,
    }

def mean_test(y_t, y_r, alpha=0.05):
    """Welch's t-test for a continuous outcome: treated group t vs reference group r."""
    m_t, m_r = y_t.mean(), y_r.mean()
    welch = stats.ttest_ind(y_t, y_r, equal_var=False)
    ci = welch.confidence_interval(confidence_level=1 - alpha)
    z_crit = stats.norm.ppf(1 - alpha / 2)

    # Relative lift: same log-ratio idea as for rates, with the spend variances
    log_ratio = np.log(m_t / m_r)
    se_log = np.sqrt(y_t.var(ddof=1) / (y_t.size * m_t**2)
                     + y_r.var(ddof=1) / (y_r.size * m_r**2))

    return {
        "treated_value": m_t,
        "reference_value": m_r,
        "abs_lift": m_t - m_r,
        "abs_ci_low": ci.low,
        "abs_ci_high": ci.high,
        "rel_lift": m_t / m_r - 1,
        "rel_ci_low": np.exp(log_ratio - z_crit * se_log) - 1,
        "rel_ci_high": np.exp(log_ratio + z_crit * se_log) - 1,
        "p_value": welch.pvalue,
    }

In [7]:
TESTS = {"visit": proportion_test, "conversion": proportion_test, "spend": mean_test}

rows = []
for outcome, test in TESTS.items():
    for treated, reference in PAIRS:
        result = test(values(outcome, treated), values(outcome, reference))
        rows.append({"outcome": outcome, "treated": treated, "reference": reference,
                     "test": test.__name__, **result})
results = pd.DataFrame(rows)

# Holm correction across the three comparisons, separately for each outcome
results["p_holm"] = results.groupby("outcome")["p_value"].transform(
    lambda p: multipletests(p, method="holm")[1]
)
results.round(3)

,outcome,treated,reference,test,treated_value,reference_value,abs_lift,abs_ci_low,abs_ci_high,rel_lift,rel_ci_low,rel_ci_high,p_value,p_holm
0,visit,mens_email,no_email,proportion_test,0.183,0.106,0.077,0.070,0.083,0.721,0.640,0.806,0.000,0.000
1,visit,womens_email,no_email,proportion_test,0.151,0.106,0.045,0.039,0.052,0.426,0.356,0.500,0.000,0.000
2,visit,mens_email,womens_email,proportion_test,0.183,0.151,0.031,0.024,0.038,0.207,0.157,0.260,0.000,0.000
3,conversion,mens_email,no_email,proportion_test,0.013,0.006,0.007,0.005,0.009,1.188,0.768,1.709,0.000,0.000
4,conversion,womens_email,no_email,proportion_test,0.009,0.006,0.003,0.001,0.005,0.543,0.230,0.936,0.000,0.000
5,conversion,mens_email,womens_email,proportion_test,0.013,0.009,0.004,0.002,0.006,0.418,0.178,0.707,0.000,0.000
6,spend,mens_email,no_email,mean_test,1.423,0.653,0.770,0.485,1.055,1.179,0.628,1.916,0.000,0.000
7,spend,womens_email,no_email,mean_test,1.077,0.653,0.424,0.169,0.680,0.650,0.218,1.236,0.001,0.002
8,spend,mens_email,womens_email,mean_test,1.423,1.077,0.345,0.033,0.658,0.321,0.027,0.699,0.030,0.030


In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

GROUP_NAMES = {"mens_email": "Men's email", "womens_email": "Women's email", "no_email": "No email"}
METRIC_NAMES = {"visit": "Visit rate", "conversion": "Conversion rate", "spend": "Revenue per customer"}

UNITS = {
    "visit": (100, "pp"),
    "conversion": (100, "pp"),
    "spend": (1, "$"),
}
COLOR = "#2a78d6"

titles = [METRIC_NAMES[o] for o in UNITS]
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=titles)

for col, (outcome, (scale, unit)) in enumerate(UNITS.items(), start=1):
    d = results[results["outcome"] == outcome]
    x, low, high = (d[c].to_numpy() * scale for c in ("abs_lift", "abs_ci_low", "abs_ci_high"))
    labels = [f"{GROUP_NAMES[t]} vs {GROUP_NAMES[r].lower()}"
              for t, r in zip(d["treated"], d["reference"])]

    fig.add_trace(go.Scatter(
        x=x,
        y=labels,
        mode="markers",
        marker=dict(size=10, color=COLOR, line=dict(width=2, color="white")),
        error_x=dict(type="data", symmetric=False, array=high - x, arrayminus=x - low,
                     thickness=2, width=0, color=COLOR),
        customdata=list(zip(low, high)),
        hovertemplate=("%{y}<br>%{x:+.2f} " + unit
                       + "<br>95% CI %{customdata[0]:.2f} to %{customdata[1]:.2f}<extra></extra>"),
        showlegend=False,
    ), row=1, col=col)

    # Zero = no effect; always keep it in view
    fig.add_vline(x=0, line=dict(width=1, color="#9b9a95"), row=1, col=col)
    fig.update_xaxes(title_text=unit, rangemode="tozero", zeroline=False, row=1, col=col)

fig.update_yaxes(autorange="reversed")  # first comparison on top
fig.update_layout(
    title="Absolute lift with 95% confidence interval",
    template="plotly_white",
    height=360,
    margin=dict(l=10, r=20, t=100, b=50),
)
fig.show()

In [10]:
# here we check if there is a difference in spending between buyers among different treatments
tdf = df.groupby("treatment").agg(conversion=("conversion", "mean"),
                                     revenue_per_customer=("spend", "mean"))
tdf["spend_per_buyer"] = tdf["revenue_per_customer"] / tdf["conversion"]
tdf.reindex(["no_email", "womens_email", "mens_email"]).round(3)

,conversion,revenue_per_customer,spend_per_buyer
treatment,,,
no_email,0.006,0.653,114.003
womens_email,0.009,1.077,121.895
mens_email,0.013,1.423,113.527


# Results  
1. Compared to the no-email group, the men email increased the revenue per emailed customer by 0.77$ on average. 
2. Compared to the women email, the men email still increased the revenue per emailed customer by 0.35$ on average. This difference is smaller (CI is between 0.03 to 0.66$), but still significant (p-value 0.03)
3. The increased revenue comes from more clients, not from bigger purchases. In fact the spending per buyer does not change between the no email group and the mens group.